# 믿:음 2.0 Base · Colab 시연용 OpenAI 호환 서버

이 노트북은 `K-intelligence/Midm-2.0-Base-Instruct`를 **4-bit로 Colab GPU에 로드**하고, 현재 가족센터 FastAPI가 호출할 수 있는 `/v1/chat/completions` API를 엽니다.

- 시연·합성 페르소나 데이터만 사용하세요. 실제 내담자 개인정보는 보내지 마세요.
- Colab 탭과 런타임이 살아 있는 동안만 동작하며, 재연결하면 터널 URL이 바뀝니다.
- 먼저 `런타임 > 런타임 유형 변경 > T4 GPU` 이상을 선택하세요.
- Colab 왼쪽의 **열쇠(Secrets)** 에 `NGROK_AUTHTOKEN`을 추가하고 노트북 액세스를 허용하세요. `HF_TOKEN`은 다운로드 제한이 생길 때만 선택적으로 추가합니다.
- ngrok 무료 계정과 authtoken은 https://dashboard.ngrok.com/get-started/your-authtoken 에서 준비합니다.


## 1. 패키지 설치
처음 한 번 수 분이 걸릴 수 있습니다. 설치 후 런타임을 재시작하라는 메시지가 나와도 우선 다음 셀을 실행해 보세요.


In [ ]:
%pip -q install -U "transformers>=4.49,<5" "accelerate>=1.2,<2" "bitsandbytes>=0.45,<1" "fastapi>=0.115,<1" "uvicorn[standard]>=0.34,<1" "ngrok>=1.4,<2" "pydantic>=2.10,<3"


## 2. GPU와 Secret 확인
`MIDM_API_KEY` Secret은 선택 사항입니다. 없으면 이 런타임 전용 키를 자동 생성해 마지막에 출력합니다.


In [ ]:
import os
import secrets
import torch
from google.colab import userdata

MODEL_ID = "K-intelligence/Midm-2.0-Base-Instruct"
SERVER_PORT = 8000
MAX_INPUT_TOKENS = 6144
MAX_OUTPUT_TOKENS = 1600

if not torch.cuda.is_available():
    raise RuntimeError("GPU가 없습니다. 런타임 > 런타임 유형 변경에서 T4 GPU 이상을 선택하세요.")

def optional_secret(name: str) -> str:
    try:
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""

NGROK_AUTHTOKEN = optional_secret("NGROK_AUTHTOKEN")
HF_TOKEN = optional_secret("HF_TOKEN") or None
MIDM_API_KEY = optional_secret("MIDM_API_KEY") or secrets.token_urlsafe(32)
if not NGROK_AUTHTOKEN:
    raise RuntimeError("Colab Secrets에 NGROK_AUTHTOKEN을 추가하고 노트북 액세스를 켜세요.")

gpu_name = torch.cuda.get_device_name(0)
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"GPU: {gpu_name}")
print(f"4-bit 계산 dtype: {compute_dtype}")
print("Secret 확인 완료 (실제 값은 표시하지 않음)")


## 3. 믿:음 Base 4-bit 로드
최초 실행은 약 23GB 원본 가중치를 내려받고 양자화하며, 환경에 따라 5~20분 정도 걸릴 수 있습니다. 완료 메시지가 나올 때까지 기다리세요.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True,
    quantization_config=quantization_config,
    torch_dtype=compute_dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model.eval()
allocated = torch.cuda.memory_allocated(0) / 1024**3
reserved = torch.cuda.memory_reserved(0) / 1024**3
print(f"모델 로드 완료 · GPU allocated {allocated:.1f}GB / reserved {reserved:.1f}GB")


## 4. OpenAI 호환 API 정의 및 로컬 서버 시작
동시 생성은 GPU 메모리 급증을 피하기 위해 1건씩 처리합니다. 입력은 6,144토큰, 출력은 1,600토큰으로 제한합니다.


In [ ]:
import asyncio
import threading
import time
import uuid
from typing import Literal

import uvicorn
from fastapi import Depends, FastAPI, Header, HTTPException
from pydantic import BaseModel, Field

app = FastAPI(title="Mi:dm 2.0 Base Colab Demo Server", version="0.1.0")
generation_lock = threading.Lock()

class ChatMessage(BaseModel):
    role: Literal["system", "user", "assistant"]
    content: str = Field(min_length=1, max_length=50000)

class ChatCompletionRequest(BaseModel):
    model: str = MODEL_ID
    messages: list[ChatMessage] = Field(min_length=1, max_length=50)
    max_tokens: int = Field(default=900, ge=1, le=MAX_OUTPUT_TOKENS)
    temperature: float = Field(default=0.35, ge=0.0, le=2.0)
    top_p: float = Field(default=0.9, gt=0.0, le=1.0)
    stream: bool = False

def require_api_key(authorization: str | None = Header(default=None)) -> None:
    expected = f"Bearer {MIDM_API_KEY}"
    if not authorization or not secrets.compare_digest(authorization, expected):
        raise HTTPException(status_code=401, detail="Invalid API key")

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID, "gpu": gpu_name}

@app.get("/v1/models", dependencies=[Depends(require_api_key)])
def models():
    return {"object": "list", "data": [{"id": MODEL_ID, "object": "model", "owned_by": "K-intelligence"}]}

def generate_sync(request: ChatCompletionRequest):
    messages = [item.model_dump() for item in request.messages]
    with generation_lock:
        batch = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )
        # Mi:dm tokenizer가 token_type_ids를 반환해도 모델에는 전달하지 않습니다.
        # 지원하는 텐서만 명시적으로 선택해 Transformers 버전 차이도 흡수합니다.
        batch = {key: value for key, value in batch.items() if key in {"input_ids", "attention_mask"}}
        input_tokens = int(batch["input_ids"].shape[-1])
        if input_tokens > MAX_INPUT_TOKENS:
            raise ValueError(f"입력이 {input_tokens}토큰입니다. {MAX_INPUT_TOKENS}토큰 이하로 줄이세요.")
        batch = {key: value.to(model.device) for key, value in batch.items()}
        generation_args = {
            "max_new_tokens": request.max_tokens,
            "do_sample": request.temperature > 0,
            "pad_token_id": tokenizer.pad_token_id,
            "eos_token_id": tokenizer.eos_token_id,
            "use_cache": True,
        }
        if request.temperature > 0:
            generation_args.update(temperature=max(0.01, request.temperature), top_p=request.top_p)
        with torch.inference_mode():
            output = model.generate(**batch, **generation_args)
        generated = output[0, input_tokens:]
        text = tokenizer.decode(generated, skip_special_tokens=True).strip()
        # greedy 생성이 첫 토큰에서 EOS로 끝나는 경우 한 번만 sampling으로 재시도합니다.
        if not text and not generation_args["do_sample"]:
            retry_args = {**generation_args, "do_sample": True, "temperature": 0.35, "top_p": 0.9}
            with torch.inference_mode():
                output = model.generate(**batch, **retry_args)
            generated = output[0, input_tokens:]
            text = tokenizer.decode(generated, skip_special_tokens=True).strip()
        if not text:
            token_preview = generated[:16].detach().cpu().tolist()
            raise ValueError(f"모델이 빈 응답을 생성했습니다. 생성 토큰: {token_preview}")
        return text, input_tokens, int(generated.shape[-1])

@app.post("/v1/chat/completions", dependencies=[Depends(require_api_key)])
async def chat_completions(request: ChatCompletionRequest):
    if request.stream:
        raise HTTPException(status_code=400, detail="이 시연 서버는 stream=false만 지원합니다.")
    try:
        text, prompt_tokens, completion_tokens = await asyncio.to_thread(generate_sync, request)
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc)) from exc
    return {
        "id": f"chatcmpl-{uuid.uuid4().hex}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": MODEL_ID,
        "choices": [{"index": 0, "message": {"role": "assistant", "content": text}, "finish_reason": "stop"}],
        "usage": {"prompt_tokens": prompt_tokens, "completion_tokens": completion_tokens, "total_tokens": prompt_tokens + completion_tokens},
    }

if "uvicorn_server" in globals():
    uvicorn_server.should_exit = True
    if "server_thread" in globals() and server_thread.is_alive():
        server_thread.join(timeout=10)
    if "server_thread" in globals() and server_thread.is_alive():
        raise RuntimeError("이전 API 서버가 아직 종료되지 않았습니다. 5초 후 이 셀만 다시 실행하세요.")
uvicorn_server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=SERVER_PORT, log_level="warning"))
server_thread = threading.Thread(target=uvicorn_server.run, daemon=True)
server_thread.start()
deadline = time.time() + 15
while not uvicorn_server.started and server_thread.is_alive() and time.time() < deadline:
    time.sleep(0.1)
if not uvicorn_server.started:
    raise RuntimeError(f"로컬 API가 포트 {SERVER_PORT}에서 시작되지 않았습니다. 서버 셀 출력을 확인하세요.")
print(f"로컬 API 시작: http://127.0.0.1:{SERVER_PORT} · 최신 서버 코드 적용됨")


## 5. 로컬 API 1차 테스트
첫 생성은 CUDA 준비 때문에 이후 요청보다 느릴 수 있습니다. 응답 내용이 출력되면 모델과 API가 정상입니다.


In [ ]:
import requests

headers = {"Authorization": f"Bearer {MIDM_API_KEY}", "Content-Type": "application/json"}
test_payload = {
    "model": MODEL_ID,
    "messages": [
        {"role": "system", "content": "너는 한국어로 간결하게 답하는 테스트 도우미다."},
        {"role": "user", "content": "연결 확인이라고 짧게 답해줘."},
    ],
    "max_tokens": 48,
    "temperature": 0.0,
    "stream": False,
}
response = requests.post(f"http://127.0.0.1:{SERVER_PORT}/v1/chat/completions", headers=headers, json=test_payload, timeout=180)
if not response.ok:
    try:
        error_body = response.json()
    except ValueError:
        error_body = response.text
    raise RuntimeError(
        f"믿:음 로컬 API HTTP {response.status_code}: {error_body}\n"
        "위의 FastAPI 서버 셀을 다시 실행한 뒤 이 테스트 셀을 재실행하세요."
    )
print(response.json()["choices"][0]["message"]["content"])


## 6. ngrok 임시 HTTPS 주소 열기
셀 출력의 `.env` 블록을 로컬 프로젝트 루트의 `.env`에 그대로 반영합니다. API 키가 있으므로 URL만 알아서는 생성 API를 호출할 수 없습니다.


In [ ]:
import inspect
import re
import requests
import ngrok

# 같은 Colab 런타임에서 이 셀을 다시 실행해도 리스너가 중복되지 않게 정리합니다.
try:
    listeners_result = ngrok.get_listeners()
    listeners = await listeners_result if inspect.isawaitable(listeners_result) else listeners_result
    for listener in listeners:
        close_result = listener.close()
        if inspect.isawaitable(close_result):
            await close_result
except Exception as cleanup_error:
    print(f"현재 런타임 리스너 정리 건너뜀: {cleanup_error}")

try:
    forward_result = ngrok.forward(SERVER_PORT, authtoken=NGROK_AUTHTOKEN)
    ngrok_listener = await forward_result if inspect.isawaitable(forward_result) else forward_result
    PUBLIC_URL = ngrok_listener.url().rstrip("/")
except ValueError as exc:
    error_text = " ".join(str(item) for item in exc.args)
    url_match = re.search(r"https://[A-Za-z0-9.-]+\.ngrok-free\.dev", error_text)
    if "ERR_NGROK_334" not in error_text or not url_match:
        raise
    candidate_url = url_match.group(0).rstrip("/")
    try:
        health_response = requests.get(
            f"{candidate_url}/health",
            headers={"ngrok-skip-browser-warning": "1"},
            timeout=15,
        )
        health_body = health_response.json() if health_response.ok else {}
    except Exception:
        health_response, health_body = None, {}
    if health_response is not None and health_response.ok and health_body.get("status") == "ok":
        PUBLIC_URL = candidate_url
        ngrok_listener = None
        print(f"이미 실행 중인 ngrok 주소를 재사용합니다: {PUBLIC_URL}")
    else:
        raise RuntimeError(
            f"이전 ngrok 주소가 계정에 남아 있지만 응답하지 않습니다: {candidate_url}\n"
            "이전 Colab 런타임을 종료하거나 ngrok 대시보드의 Endpoints에서 해당 주소를 중지한 뒤 이 셀을 다시 실행하세요."
        ) from exc
OPENAI_BASE_URL = f"{PUBLIC_URL}/v1"

print("\n===== 로컬 프로젝트 .env에 넣을 값 =====")
print("AI_PROVIDER=internal_openai")
print(f"INTERNAL_LLM_BASE_URL={OPENAI_BASE_URL}")
print(f"INTERNAL_LLM_MODEL={MODEL_ID}")
print(f"INTERNAL_LLM_API_KEY={MIDM_API_KEY}")
print("LLM_REQUEST_TIMEOUT=240")
print("LLM_HEALTH_TIMEOUT=12")
print("=========================================\n")
print("중요: Colab 런타임이 재연결되면 이 셀을 다시 실행하고 새 URL로 .env를 갱신하세요.")


## 7. 외부 주소 최종 테스트
여기까지 성공하면 Windows의 현재 FastAPI도 같은 주소를 호출할 수 있습니다.


In [ ]:
public_headers = {**headers, "ngrok-skip-browser-warning": "1"}
models_response = requests.get(f"{OPENAI_BASE_URL}/models", headers=public_headers, timeout=30)
models_response.raise_for_status()
print("외부 연결 정상:", models_response.json()["data"][0]["id"])


## 8. Windows 앱 연결
1. 위 출력값을 프로젝트 루트 `.env`에 붙여 넣습니다. 기존 키가 있으면 해당 줄을 교체합니다.
2. 실행 중인 **백엔드 PowerShell에서 `Ctrl+C`** 후 프로젝트 루트에서 다시 실행합니다.
```powershell
.\.venv\Scripts\python.exe -m pip install -r backend\requirements.txt
.\.venv\Scripts\python.exe -m uvicorn backend.app.main:app --host 127.0.0.1 --port 8100
```
3. 프론트엔드는 재시작하지 않아도 됩니다. `http://127.0.0.1:3000/training` 또는 상담 코파일럿 화면을 새로고침합니다.
4. 상단에 **믿:음 연결 정상**이 보이면 완료입니다. 오프라인이면 이 노트북의 6·7번 셀과 `.env` URL을 확인하세요.

Colab 탭을 닫거나 런타임이 종료되면 기존 화면은 유지되지만 새 AI 응답 생성은 실패합니다. 시연 시작 전에 7번 셀을 한 번 실행해 상태를 확인하세요.


## 9. PaddleOCR-VL 1.6 격리 사이드카 설치·실행
Mi:dm은 Transformers 4.x를 유지하고, PaddleOCR-VL 1.6은 Transformers 5 전용 가상환경과 별도 프로세스에서 실행합니다. 두 프로세스는 같은 Colab GPU를 공유하며 다음 프록시 셀의 공용 잠금으로 동시에 GPU를 사용하지 않습니다.


In [ ]:
import json, os, shutil, subprocess, sys, textwrap, time
from pathlib import Path

OCR_MODEL_ID = "PaddlePaddle/PaddleOCR-VL-1.6"
OCR_CONTRACT_VERSION = "soap-content-v2"
OCR_PORT = 8121
OCR_ENV = Path("/content/paddleocr_vl_env")
OCR_PYTHON = OCR_ENV / "bin/python"
OCR_PIP = OCR_ENV / "bin/pip"
OCR_SCRIPT = Path("/content/paddle_ocr_vl_sidecar.py")

if not OCR_PYTHON.exists() or not OCR_PIP.exists():
    # Colab Python 3.12 이미지에는 ensurepip/python3-venv가 빠진 경우가 있어 virtualenv를 사용합니다.
    if OCR_ENV.exists():
        shutil.rmtree(OCR_ENV)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "virtualenv>=20.28,<21"])
    subprocess.check_call([
        sys.executable, "-m", "virtualenv", "--system-site-packages", str(OCR_ENV)
    ])
subprocess.check_call([
    str(OCR_PYTHON), "-m", "pip", "install", "-q", "-U",
    "transformers>=5,<6", "accelerate>=1.2,<2", "fastapi>=0.115,<1",
    "uvicorn[standard]>=0.34,<1", "pydantic>=2.10,<3", "Pillow>=10,<13",
])

sidecar_source = r'''
import base64
import io
import threading

import torch
import uvicorn
from fastapi import FastAPI, HTTPException
from PIL import Image
from pydantic import BaseModel, Field
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "PaddlePaddle/PaddleOCR-VL-1.6"
MAX_NEW_TOKENS = 512
CONTRACT_VERSION = "soap-content-v2"
OCR_PROMPT = (
    "OCR:\n이미지에 실제로 보이는 내용을 원문 그대로 전사하세요. 요약하거나 추론하거나 맞춤법을 고치지 마세요. "
    "SOAP 양식이면 인쇄된 작성 안내, 영문 항목명, 예시 불릿은 제외하고 각 답변 칸에 작성된 내용만 "
    "S:, O:, A:, P: 네 줄로 구분해 출력하세요. 다른 문서라면 작성된 내용을 빠짐없이 전사하세요. "
    "설명이나 마크다운은 덧붙이지 마세요."
)
app = FastAPI(title="PaddleOCR-VL 1.6 Colab sidecar")
ocr_lock = threading.Lock()
processor = None
model = None
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

class OCRRequest(BaseModel):
    images: list[str] = Field(min_length=1, max_length=8)

def load_model():
    global processor, model
    if model is None:
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA GPU가 필요합니다.")
        processor = AutoProcessor.from_pretrained(MODEL_ID)
        model = AutoModelForImageTextToText.from_pretrained(
            MODEL_ID, torch_dtype=dtype, low_cpu_mem_usage=True
        ).to("cuda").eval()

def read_image(image):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image.convert("RGB")},
        {"type": "text", "text": OCR_PROMPT},
    ]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    )
    moved = {}
    for key, value in inputs.items():
        if hasattr(value, "is_floating_point") and value.is_floating_point():
            moved[key] = value.to(device="cuda", dtype=dtype)
        elif hasattr(value, "to"):
            moved[key] = value.to("cuda")
        else:
            moved[key] = value
    input_length = moved["input_ids"].shape[-1]
    with torch.inference_mode():
        output = model.generate(**moved, do_sample=False, max_new_tokens=MAX_NEW_TOKENS)
    return processor.decode(output[0][input_length:], skip_special_tokens=True).strip()

@app.get("/ocr/status")
def status():
    gpu = bool(torch.cuda.is_available())
    return {
        "status": "ok" if gpu else "unavailable",
        "available": gpu, "gpu_available": gpu, "model": MODEL_ID,
        "contract_version": CONTRACT_VERSION,
        "loaded": model is not None,
        "detail": "Colab PaddleOCR-VL 1.6 사이드카가 준비되었습니다." if gpu else "CUDA GPU가 없습니다.",
    }

@app.post("/ocr")
def ocr(request: OCRRequest):
    try:
        images = [Image.open(io.BytesIO(base64.b64decode(value, validate=True))).convert("RGB") for value in request.images]
    except Exception as exc:
        raise HTTPException(status_code=400, detail=f"이미지 데이터 오류: {exc}") from exc
    try:
        with ocr_lock:
            load_model()
            texts = [read_image(image) for image in images]
    except RuntimeError as exc:
        raise HTTPException(status_code=503, detail=str(exc)) from exc
    return {"model": MODEL_ID, "texts": texts}

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=8121, log_level="warning")
'''
OCR_SCRIPT.write_text(textwrap.dedent(sidecar_source), encoding="utf-8")

def ocr_sidecar_ready():
    try:
        response = requests.get(f"http://127.0.0.1:{OCR_PORT}/ocr/status", timeout=3)
        return response.ok and response.json().get("contract_version") == OCR_CONTRACT_VERSION
    except Exception:
        return False

if not ocr_sidecar_ready():
    if "ocr_sidecar_process" in globals() and ocr_sidecar_process.poll() is None:
        ocr_sidecar_process.terminate()
        ocr_sidecar_process.wait(timeout=10)
    ocr_sidecar_log = open("/content/paddle_ocr_sidecar.log", "a", encoding="utf-8")
    sidecar_env = {**os.environ, "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
    ocr_sidecar_process = subprocess.Popen(
        [str(OCR_PYTHON), str(OCR_SCRIPT)], stdout=ocr_sidecar_log, stderr=subprocess.STDOUT, env=sidecar_env
    )
    deadline = time.time() + 90
    while time.time() < deadline and not ocr_sidecar_ready():
        if ocr_sidecar_process.poll() is not None:
            raise RuntimeError(Path("/content/paddle_ocr_sidecar.log").read_text(encoding="utf-8")[-3000:])
        time.sleep(1)
if not ocr_sidecar_ready():
    raise RuntimeError("PaddleOCR-VL 사이드카가 90초 안에 시작되지 않았습니다.")
print("PaddleOCR-VL 격리 사이드카 준비 완료 · 모델은 첫 OCR 요청에서 한 번 로드됩니다.")


## 10. 기존 ngrok 서버에 OCR 프록시 등록·확인
같은 API 키와 ngrok 주소를 사용합니다. OCR 중에는 공용 GPU 잠금을 잡아 Mi:dm 생성과 동시에 실행되지 않게 합니다.


In [ ]:
class OCRProxyRequest(BaseModel):
    images: list[str] = Field(min_length=1, max_length=8)

ocr_paths = {"/v1/ocr", "/v1/ocr/status"}
app.router.routes = [route for route in app.router.routes if getattr(route, "path", None) not in ocr_paths]

@app.get("/v1/ocr/status", dependencies=[Depends(require_api_key)])
def ocr_proxy_status():
    try:
        response = requests.get(f"http://127.0.0.1:{OCR_PORT}/ocr/status", timeout=10)
        response.raise_for_status()
        return response.json()
    except Exception as exc:
        raise HTTPException(status_code=503, detail=f"OCR 사이드카 상태 확인 실패: {exc}") from exc

@app.post("/v1/ocr", dependencies=[Depends(require_api_key)])
def ocr_proxy(request: OCRProxyRequest):
    with generation_lock:
        try:
            response = requests.post(
                f"http://127.0.0.1:{OCR_PORT}/ocr", json=request.model_dump(), timeout=300
            )
        except Exception as exc:
            raise HTTPException(status_code=503, detail=f"OCR 사이드카 연결 실패: {exc}") from exc
    if not response.ok:
        raise HTTPException(status_code=response.status_code, detail=response.text[:1000])
    return response.json()

ocr_status_response = requests.get(
    f"{PUBLIC_URL}/v1/ocr/status", headers=public_headers, timeout=30
)
ocr_status_response.raise_for_status()
print("외부 PaddleOCR 상태 정상:", ocr_status_response.json())
print("OCR_PROVIDER=paddleocr_vl_http")
print(f"INTERNAL_OCR_URL={PUBLIC_URL}/v1/ocr")
print(f"INTERNAL_OCR_API_KEY={MIDM_API_KEY}")
print("OCR_REQUEST_TIMEOUT=300")
print("OCR_HEALTH_TIMEOUT=12")
print("OCR_REMOTE_BATCH_SIZE=4")
